# User Mapping Analysis

Created On: 7/2/2025

Created By: Juli Hirt

Objective of this notebook: Analyze players' mapping between their terms to the 17 categories, including 1 (term) to 1 (category) mapping and 1 (term) to many (categories) mapping.
This helps us identify Related Terms for the categories. Enriching categories into a taxonomy like #1.

Category - Term (with term counts) display

Term - Category display


## Imports

In [ ]:
import pandas as pd
import numpy as np 
import ast
import string

from nltk.stem import PorterStemmer
from IPython.display import display


stemmer = PorterStemmer()



# Load Data

In [ ]:
file_name = 'all_terms_final.xlsx'
df = pd.read_excel(f'Data/Outbound/{file_name}', sheet_name='Raw Data')

In [ ]:
df.head(3)

In [ ]:
df.shape

## Prepare Data Frame

In [ ]:
# -------------------
# Step 1: Process and group terms
# -------------------

term_summary = {}
all_cats = set()

for idx, row in df.iterrows():
    text = row['QX_X.CleanedMoodTerm'].strip()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    stems = [stemmer.stem(token.lower()) for token in tokens]
    stem = stems[0] if stems else ''

    categories = [row['QX_X.UserCategoryMapping'].strip()]
    if pd.notna(row['QX#1_X.MultipleCategoryMapping']):
        try:
            multi = ast.literal_eval(row['QX#1_X.MultipleCategoryMapping'])
            categories += multi
        except:
            print(idx, "Error parsing multiple categories")

    categories = list(set(categories))
    all_cats.update(categories)

    if text not in term_summary:
        # Initialize with zero for known categories so far
        term_summary[text] = {
            'Stem': stem,
            'Categories': categories,
            'Count': 1,
            'CategoryCounts': {cat: 1 if cat in categories else 0 for cat in categories}
        }
    else:
        term_summary[text]['Count'] += 1
        for cat in categories:
            if cat in term_summary[text]['CategoryCounts']:
                term_summary[text]['CategoryCounts'][cat] += 1
            else:
                term_summary[text]['CategoryCounts'][cat] = 1

# After all rows: make sure each CategoryCounts has *all* categories
for term_info in term_summary.values():
    for cat in all_cats:
        if cat not in term_info['CategoryCounts']:
            term_info['CategoryCounts'][cat] = 0

In [ ]:
# -------------------
# Step 2: Build term-category matrix (binary indicators)
# -------------------

cat_dict = {}
for term, info in term_summary.items():
    cat_dict[term] = {}
    for cat in all_cats:
        cat_dict[term][cat] = 1 if cat in info['Categories'] else 0

df_terms = pd.DataFrame.from_dict(cat_dict, orient='index')
df_terms = df_terms.fillna(0).astype(int)

In [ ]:
# -------------------
# Step 3: Build stem-to-terms mapping
# -------------------

stem_to_terms = {}
for term, info in term_summary.items():
    stem = info['Stem']
    if stem not in stem_to_terms:
        stem_to_terms[stem] = []
    stem_to_terms[stem].append(term)

In [ ]:
# -------------------
# Step 4: Build grouped DataFrame with header rows and correct counts
# -------------------

new_rows = []

# Sort stems alphabetically
for stem in sorted(stem_to_terms.keys()):
    terms = stem_to_terms[stem]

    # Stem total count
    stem_count = sum(term_summary[term]['Count'] for term in terms if term in term_summary)

    # Category totals
    totals = pd.Series(0, index=df_terms.columns)
    for term in terms:
        if term in term_summary:
            for col in df_terms.columns:
                totals[col] += term_summary[term]['CategoryCounts'][col]

    # Header row
    header_row = {'Term': f"Stem: {stem}", 'RowType': 'Stem', 'Count': stem_count}
    for col in df_terms.columns:
        header_row[col] = totals[col]
    new_rows.append(header_row)

    # Child term rows
    for term in terms:
        if term in term_summary:
            row_data = {'Term': f"   → {term}", 'RowType': 'Term', 'Count': term_summary[term]['Count']}
            for col in df_terms.columns:
                row_data[col] = term_summary[term]['CategoryCounts'][col]
            new_rows.append(row_data)



In [ ]:
# -------------------
# Step 5: Create final DataFrame and display
# -------------------

df_grouped = pd.DataFrame(new_rows)

# Sort category columns alphabetically
sorted_cats = sorted([col for col in df_terms.columns])

# Build final column order
cols = ['Term', 'RowType', 'Count'] + sorted_cats
df_grouped = df_grouped[cols]

display(df_grouped)

In [ ]:
df_grouped = df_grouped.to_excel('Data/Outbound/term_summary.xlsx', index=False)